<a href="https://colab.research.google.com/github/HerinePamela/Notebook/blob/main/GEE_Colab_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [36]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='stoked-mapper-346107')

In [24]:
Provinces = geemap.shp_to_ee('/content/drive/MyDrive/Eritrea/Provinces.shp')

In [39]:
# Sentinel-2 scaling (correct for S2_SR)
def applyScaleFactors(image):
    optical = image.select(['B.*']).divide(10000)
    return image.addBands(optical, None, True)

# LOAD SENTINEL-2 DATA
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterDate('2015-01-01', '2016-12-31')
    .filterBounds(Provinces)
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
    .map(applyScaleFactors)
    .median())

# True Color visualization
vis_params_true_color = {
    'bands': ['B4', 'B3', 'B2'],
    'min': 0,
    'max': 0.3,
    'gamma': 1.3
}

Map = geemap.Map()

Map.add_basemap('SATELLITE')  # Google Satellite
Map.centerObject(Provinces, 8)

Map.addLayer(image.clip(Provinces), vis_params_true_color, 'True Color')
Map

Map(center=[15.69801053808507, 38.5143741979578], controls=(WidgetControl(options=['position', 'transparent_bg…

In [46]:
# training points
water = geemap.shp_to_ee('/content/drive/MyDrive/Eritrea/Water.shp')
built_up = geemap.shp_to_ee('/content/drive/MyDrive/Eritrea/Built_Up.shp')
bareland = geemap.shp_to_ee('/content/drive/MyDrive/Eritrea/Bareland.shp')
farmlands = geemap.shp_to_ee('/content/drive/MyDrive/Eritrea/Farmlands.shp')
vegetation = geemap.shp_to_ee('/content/drive/MyDrive/Eritrea/Vegetation.shp')

# Asign labels
water = water.map(lambda f: f.set('class', 0))
built_up = built_up.map(lambda f: f.set('class', 1))
bareland = bareland.map(lambda f: f.set('class', 2))
farmlands = farmlands.map(lambda f: f.set('class', 3))
vegetation = vegetation.map(lambda f: f.set('class', 4))

# Merge points
training_points = water.merge(built_up)\
                       .merge(bareland)\
                       .merge(farmlands)\
                       .merge(vegetation)

# Prepare sentinel data
image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
         .filterBounds(Provinces)
         .filterDate('2015-01-01', '2015-12-31')
         .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20))
         .median()
         .divide(10000))

bands = ['B2','B3','B4','B8','B11','B12']



In [47]:
training = image.select(bands).sampleRegions(
    collection = training_points,
    properties = ['class'],
    scale = 10
)

In [48]:
classifier = ee.Classifier.smileRandomForest(100).train(
    features = training,
    classProperty = 'class',
    inputProperties = bands
)

In [49]:
classified = image.select(bands).classify(classifier)

In [50]:
palette = ['blue','red','yellow','orange','green']

Map.addLayer(classified, {'min':0,'max':4,'palette':palette}, 'LULC')
Map

Map(bottom=4008.0, center=[16.804541076383455, 37.55126953125001], controls=(WidgetControl(options=['position'…

In [41]:
# label = lc
# bands = ['B3', 'B5', 'B8']

# # sample the input images
# sample = image.addBands(lc).sample(**(
#     'region':image.Provinces(),
#     'scale':30,
#     'numPixels':5000,
#     'seed':1

# ))

# sample = sample.randomColumn()
# split = 0.7
# training = sample.filter(ee.Filter.lt('random', split))
# validation = sample.filter(ee.Filter.gte('random', split))

# # train
# classifier = ee.Classifier.smileRandomForest(10).train(**{
#     'features': training,
#     # 'classProperty': label,
#     'inputProperties': bands})

# model = image.classsify(classifier)

SyntaxError: invalid syntax (2319711733.py, line 6)

In [ ]:
# # Define dictionary
# dict = {
#     'names':[
#         'Built_Up',
#         'Water',
#         'Farmlands',
#         'Bareland',
#         'Vegetation'
#     ],
#     'colors':[
#         '#ee2601',
#         '#1db7ff',
#         '#aef2a4',
#         '#d0850c',
#         '#0c8d01'
#     ]
# }

In [ ]:
# # Add Landcover to the map
# Map.addLayer(lc.clip(image.Provinces()), {'min':1, 'max':9, 'palette': dict('colors')}, "Esrilulc")
# Map.addLayer(model, {'min':0, 'max':4, 'palette':dict['colors']}, 'Landcover')
# Map.centerObject(image,8)
# Map

In [51]:
# Export

Config = {
    'folder':'landcover',
    'scale':30,
    'region':image.Provinces(),
    'fileFormat':'GeoTIFF'
}
task = ee.batch.Export.image.toDrive(image, **Config)
task.start()

AttributeError: 'Image' object has no attribute 'Provinces'